# 10.4 Random Search

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 10: Model Optimization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-10-model-optimization/10.04_random-search.ipynb)

Companion code for Section 10.4 of the book. Run the cells in order from top to bottom.

**Example 10.4.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import uniform, randint, loguniform
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import time
import warnings; warnings.filterwarnings('ignore')

data = load_breast_cancer()
X, y = data.data, data.target

# ── Random Search with continuous distributions ───────────────────
param_dist = {
    'n_estimators':      randint(50, 500),        # Uniform integer
    'max_depth':         [None, 3, 5, 7, 10, 15, 20],
    'max_features':      uniform(0.1, 0.9),       # Uniform float [0.1, 1.0]
    'min_samples_split': randint(2, 20),
    'min_samples_leaf':  randint(1, 10),
    'bootstrap':         [True, False],
}

# ── Compare Grid vs Random at equal budget ────────────────────────
results_comparison = {}

for n_iter in [10, 20, 50, 100]:
    t0 = time.time()
    rs = RandomizedSearchCV(
        RandomForestClassifier(random_state=42, n_jobs=-1),
        param_dist, n_iter=n_iter,
        cv=5, scoring='roc_auc', n_jobs=-1, random_state=42
    )
    rs.fit(X, y)
    t1 = time.time()
    results_comparison[f'Random({n_iter})'] = {
        'score': rs.best_score_, 'time': t1-t0,
        'params': rs.best_params_
    }

print('=== Random Search Results ===')
print(f'{'Method':<16} {'Best AUC':>10} {'Time(s)':>10}')
print('='*40)
for method, result in results_comparison.items():
    print(f'{method:<16} {result["score"]:>10.4f} {result["time"]:>10.2f}')

# Best random search
rs_best = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_dist, n_iter=100,
    cv=5, scoring='roc_auc', n_jobs=-1, random_state=42
)
rs_best.fit(X, y)
print(f'\nBest params (n_iter=100):')
for k, v in rs_best.best_params_.items():
    print(f'  {k:<22}: {v}')

# ── Gradient Boosting Random Search ──────────────────────────────
gb_param_dist = {
    'n_estimators':   randint(100, 500),
    'learning_rate':  loguniform(0.01, 0.3),   # Log-uniform [0.01, 0.3]
    'max_depth':      randint(2, 8),
    'subsample':      uniform(0.5, 0.5),        # [0.5, 1.0]
    'max_features':   uniform(0.3, 0.7),        # [0.3, 1.0]
}

rs_gb = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=42),
    gb_param_dist, n_iter=50,
    cv=5, scoring='roc_auc', n_jobs=-1, random_state=42
)
rs_gb.fit(X, y)
print(f'\nGradient Boosting Random Search (n=50):')
print(f'  Best AUC: {rs_gb.best_score_:.4f}')
print(f'  Best params: {rs_gb.best_params_}')

# ── Visualise: search landscape ───────────────────────────────────
import pandas as pd
cv_df = pd.DataFrame(rs_best.cv_results_)
cv_df['rank'] = cv_df['rank_test_score']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Random Search Analysis', fontsize=14, fontweight='bold')

ax = axes[0]
ax.scatter(cv_df['param_n_estimators'], cv_df['mean_test_score'],
          c=cv_df['mean_test_score'], cmap='RdYlGn', s=40, alpha=0.7)
ax.set_xlabel('n_estimators'); ax.set_ylabel('CV AUC')
ax.set_title('n_estimators vs AUC'); ax.grid(alpha=0.3)

ax = axes[1]
ax.scatter(cv_df['param_max_features'], cv_df['mean_test_score'],
          c=cv_df['mean_test_score'], cmap='RdYlGn', s=40, alpha=0.7)
ax.set_xlabel('max_features'); ax.set_ylabel('CV AUC')
ax.set_title('max_features vs AUC'); ax.grid(alpha=0.3)

ax = axes[2]
scores_sorted = sorted(cv_df['mean_test_score'], reverse=True)
ax.plot(range(1, len(scores_sorted)+1), scores_sorted, 'b-o', ms=5, lw=2)
ax.set_xlabel('Trial (ranked)'); ax.set_ylabel('CV AUC')
ax.set_title(f'Best Score per Trial\n(Best={rs_best.best_score_:.4f})')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('random_search.png', dpi=150, bbox_inches='tight')
plt.show()